---
format:
  html:
    code-fold: true
execute:
  echo: true
---

[← Back to article notes](../index.qmd)

# Implementation notebook

- Title: Score-Based Physics-Informed Neural Networks for High-Dimensional Fokker-Planck Equations
- Authors: Hu, Zhang, Karniadakis, Kawaguchi
- Year:    2024

In [109]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

## Score-PINN

The score-PINN is a neural network that takes $\textbf{x},t$ and returns $\nabla_x\log p_t(\textbf{x})$. We call it $\textbf{s}_\theta$ and to train it we work on the optimization problem:
**Denoising Score Matching**
$$
    \max_{t}\mathbb{E}_{\textbf{x}\sim\textrm{SDE}, \textbf{z}_\varepsilon\sim\mathcal{N}\left(\textbf{0},\varepsilon\textrm{Id}\right)}\left[\left\|\textbf{s}_\theta\left(\textbf{x} + \textbf{z}_\varepsilon,t\right) + \frac{\textbf{z}_\varepsilon}{\varepsilon}\right\|^2\right]
$$
So to compute the loss function, we need a SDE simulator of:
$$
    \textrm{d}\textbf{X}_t = \textbf{f}\textrm{d}t + \sigma\textrm{d}\textbf{W}_t
$$
In this experiment we use Euler-Maruyama scheme:
$$
    \textbf{X}_{n+1} = \textbf{X}_n + \textbf{f}_n\Delta t + \sigma_{n}\Delta\textbf{W}_n
$$
where $\Delta \textbf{W}_n\sim\mathcal{N}\left(0,\Delta t\textrm{Id}\right)$.

### Denoising Score Matching
A work of @JMLR:v6:hyvarinen05a shows that for all $t$ the score-matching minimizes:
$$
    \mathbb{E}_{\textbf{x}\sim\textrm{SDE}}\left[\frac12\left\|\textbf{s}_\theta\left(\textbf{x},t\right)\right\|^2 + \nabla_\textbf{x}\cdot\textbf{s}_\theta\left(\textbf{x},t\right)\right]
$$
The minimum value is the opposite of the Fisher information $\mathbb{E}\left[\frac12\left\|\nabla_\textbf{x}\log p(x)\right\|^2\right]$.

In high-dimensional case, the computation of the divergence is expensive. So in the paper @10.1162/NECO_a_00142 the authors introduce the denoising score matching like a simple way to simplify this computation. At first we use the ralation:
$$
\begin{align*}
    \nabla_{\tilde{\textbf{x}}}\log q_\varepsilon\left(\tilde{\textbf{x}}\middle| \textbf{x}\right) = -\frac{\tilde{x}-x}{\varepsilon}=-\frac{z_\varepsilon}{\varepsilon}
\end{align*}
$$
where $q_\sigma\left(\tilde{\textbf{x}}\middle| \textbf{x}\right)$ is the gaussian multivariate density with standard deviation $\sigma$.
$$
\begin{align*}
    \mathbb{E}_{\textbf{x}\sim\textrm{SDE}, \textbf{z}_\varepsilon\sim\mathcal{N}\left(\textbf{0},\varepsilon\textrm{Id}\right)}\left[\frac12\left\|\textbf{s}_\theta\left(\textbf{x} + \textbf{z}_\varepsilon,t\right) + \frac{\textbf{z}_\varepsilon}{\varepsilon}\right\|^2\right] &= \mathbb{E}_{\textbf{x}\sim\textrm{SDE}, \textbf{z}_\varepsilon\sim\mathcal{N}\left(\textbf{0},\varepsilon\textrm{Id}\right)}\left[\frac12\left\|\textbf{s}_\theta\left(\textbf{x} + \textbf{z}_\varepsilon,t\right)\right\|^2+\textbf{s}_\theta\left(\textbf{x} + \textbf{z}_\varepsilon,t\right)\cdot\frac{\textbf{z}_\varepsilon}{\varepsilon}+\frac12\left\|\frac{\textbf{z}_\varepsilon}{\varepsilon}\right\|^2\right] \\
    &= \mathbb{E}_{q_{\varepsilon}\left(\tilde{\textbf{x}}\middle|\textbf{x}\right)}\left[\frac12\left\|\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\right\|^2+\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\cdot\frac{\tilde{\textbf{x}}-\textbf{x}}{\varepsilon}+\frac12\left\|\frac{\tilde{\textbf{x}}-\textbf{x}}{\varepsilon}\right\|^2\right] \\
    &= \mathbb{E}_{q_{\varepsilon}\left(\tilde{\textbf{x}}\middle|\textbf{x}\right)}\left[\frac12\left\|\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\right\|^2+\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\cdot\frac{\tilde{\textbf{x}}-\textbf{x}}{\varepsilon}\right] + \mathcal{O}_p\left(n\varepsilon^{-1}\right)\\
    \mathbb{E}_{q_{\varepsilon}\left(\tilde{\textbf{x}}\middle|\textbf{x}\right)}\left[-\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\cdot\nabla_{\tilde{\textbf{x}}}\log q_\varepsilon\left(\tilde{\textbf{x}}\middle| \textbf{x}\right)\right] &= -\int \textrm{d}\tilde{\textbf{x}}q_{\varepsilon}\left(\tilde{\textbf{x}}\middle|\textbf{x}\right)\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\cdot\nabla_{\tilde{\textbf{x}}}\log q_\varepsilon\left(\tilde{\textbf{x}}\middle| \textbf{x}\right)\\
    &= -\int \textrm{d}\tilde{\textbf{x}}\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\cdot\nabla_{\tilde{\textbf{x}}} q_\varepsilon\left(\tilde{\textbf{x}}\middle| \textbf{x}\right)\\
    &= \int \textrm{d}\tilde{\textbf{x}}q_\varepsilon\left(\tilde{\textbf{x}}\middle| \textbf{x}\right)\nabla_{\tilde{\textbf{x}}}\cdot\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right) - \int_{\partial\mathbb{R}^n} \textrm{d}S\left(\tilde{\textbf{x}}\right)q_\varepsilon\left(\tilde{\textbf{x}}\middle| \textbf{x}\right)\left(\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\cdot\textbf{n}\right)\\
    &= \mathbb{E}_{q_\varepsilon\left(\tilde{\textbf{x}}\middle| \textbf{x}\right)}\left[\nabla_{\tilde{\textbf{x}}}\cdot\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\right]
\end{align*}
$$
where we use the assumption that $q_\varepsilon\left(\tilde{\textbf{x}}\middle| \textbf{x}\right)\ll\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)$ as $\left\|\tilde{\textbf{x}}\right\|\to\infty$. So:
$$
\begin{align*}
    \mathbb{E}_{\textbf{x}\sim\textrm{SDE}, \textbf{z}_\varepsilon\sim\mathcal{N}\left(\textbf{0},\varepsilon\textrm{Id}\right)}\left[\frac12\left\|\textbf{s}_\theta\left(\textbf{x} + \textbf{z}_\varepsilon,t\right) + \frac{\textbf{z}_\varepsilon}{\varepsilon}\right\|^2\right] = \mathbb{E}_{q_\varepsilon\left(\tilde{\textbf{x}}\middle| \textbf{x}\right)}\left[\frac12\left\|\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\right\|^2+\nabla_{\tilde{\textbf{x}}}\cdot\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\right] + \mathcal{O}_p\left(n\varepsilon^{-1}\right)\\
    \lim_{\varepsilon\to0} \mathbb{E}_{q_\varepsilon\left(\tilde{\textbf{x}}\middle| \textbf{x}\right)}\left[\frac12\left\|\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\right\|^2+\nabla_{\tilde{\textbf{x}}}\cdot\textbf{s}_\theta\left(\tilde{\textbf{x}},t\right)\right] = \mathbb{E}_{\textbf{x}\sim\textrm{SDE}}\left[\frac12\left\|\textbf{s}_\theta\left(\textbf{x},t\right)\right\|^2+\nabla_{\textbf{x}}\cdot\textbf{s}_\theta\left(\textbf{x},t\right)\right]
\end{align*}
$$
So we know that for each optimization problem the loss function converges to the ideal loss function. In other words, it is like an Asymptotic Preserving property. Howevere, we have to remark that the theoretical optimal solution of the relaxation problem converges to the limit solution (the pure score-matching), in the case of the PINN (vanilla approach) the target's noise diverges. So the quality of the training and the optimal theoretical PINN does not converges to the optimal theoretical PINN of the limit loss. 

In [110]:
#| code-summary: "Score-PINN model"

class ScoreNetwork(nn.Module):
    def __init__(self, spatial_dim, hidden_dim):
        super().__init__()
        input_dim = spatial_dim + 1
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, spatial_dim) 
        )

    def forward(self, input_tensor):
        score = self.net(input_tensor)
        return score

In this implementation we have to compute the score for each frame.
Here the following loss is implemented:
$$
\mathbb{E}_{p(t)q_\varepsilon\left(\tilde{\textbf{x}}\middle|\textbf{x}\right)}\left[\frac12\left\|\textbf{s}_\theta\left(\textbf{x} + \textbf{z}_\varepsilon,t\right) + \frac{\textbf{z}_\varepsilon}{\varepsilon}\right\|^2\right]
$$

In [111]:
#| code-summary: "Score matching loss function"
def score_matching_loss(score_net, x_samples, t_samples, noise_std):
    noise = torch.randn_like(x_samples)
    x_perturbed = x_samples + noise * noise_std
    net_input = torch.cat([x_perturbed, t_samples], dim=-1)
    predicted_score = score_net(net_input)
    target_score = -noise / noise_std
    loss = torch.mean(torch.sum((predicted_score - target_score)**2, dim=-1))
    return loss - x_samples.shape[-1]/noise_std**2

In [112]:
#| code-summary: "Euler-Maruyama sampler"

def drift_function(x: torch.Tensor, t: torch.Tensor):
    # x shape (N,d)
    if x.ndim == 1:
        x = x.unsqueeze(0)
    # t shape (N,)
    if t.ndim == 0:
        t = t.unsqueeze(0)
    return - 4 * (torch.sum(x**2, dim=-1, keepdim=True) - 1) * x
def sigma_function(x: torch.Tensor, t: torch.Tensor):
    # x shape (N,d)
    if x.ndim == 1:
        x = x.unsqueeze(0)
    # t shape (N,)
    if t.ndim == 0:
        t = t.unsqueeze(0)
    N, d = x.shape
    return 1.5*torch.eye(d, device=x.device).repeat(N, 1, 1)

def euler_maruyama_sampler(x_init: torch.Tensor, t_init: float, t_final: float, n_steps: int):
    dt = (t_final - t_init) / n_steps

    x_samples = torch.empty((n_steps, *x_init.shape), device=x_init.device)
    t_samples = torch.linspace(t_init, t_final, n_steps, device=x_init.device)
    dW = torch.randn((n_steps, *x_init.shape), device=x_init.device) * torch.sqrt(torch.tensor(dt))

    x = x_init.detach().clone()
    for i in range(n_steps):
        x_samples[i] = x
        drift = drift_function(x, t_samples[i])
        sigma = sigma_function(x, t_samples[i])
        x = x + drift * dt + (sigma @ dW[i].unsqueeze(-1)).squeeze(-1)

    return x_samples, t_samples

In [113]:
#| code-summary: "SDE simulation"
#| echo: false
"""
N = 1000
n_steps = 1000
x_init = torch.randn(N, 2)*0.5
x_samples, t_samples = euler_maruyama_sampler(x_init, 0.0, 5.0, n_steps)

fig, ax = plt.subplots(figsize=(6, 4))
points = ax.scatter(x_samples[:, 0], x_samples[:, 1], s=1, alpha=0.5, color="blue", label="SDE Samples")
title = ax.text(0.5, 1.02, "", transform=ax.transAxes, ha="center")
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$y$")
ax.legend(loc="upper right", fontsize=8)
ax.grid(True, alpha=0.3)

fps = 30
frames = fps * (t_samples[-1]-t_samples[0]).item()
ratio = max(len(x_samples), frames) / frames

x_subsamples = x_samples[::int(ratio),:].detach().cpu().numpy()  # downsample to match the desired number of frames
Dt = (t_samples[-1]-t_samples[0]).item() / x_subsamples.shape[0]

def init():
    points.set_offsets(x_subsamples[0])
    title.set_text("")
    return points, title

def update(frame):
    points.set_offsets(x_subsamples[frame])
    title.set_text(rf"time $t = {frame * Dt:.2f}\,\mathrm{{s}}$")
    return points, title

ani = FuncAnimation(
    fig,
    update,
    frames=len(x_subsamples),
    init_func=init,
    interval=Dt*1000,  # interval in milliseconds
    blit=True
)

ani.save("sde_simulation.gif", writer=PillowWriter(fps=fps), dpi=200)
plt.close(fig)
"""
pass

<>:33: SyntaxWarning: invalid escape sequence '\,'
<>:33: SyntaxWarning: invalid escape sequence '\,'
C:\Users\Stefano\AppData\Local\Temp\ipykernel_9212\2741990679.py:33: SyntaxWarning: invalid escape sequence '\,'
  title.set_text(rf"time $t = {frame * Dt:.2f}\,\mathrm{{s}}$")


<img src="sde_simulation.gif" alt="SDE simulation animation" width="600">

In [114]:
#| code-summary: "Training loop"

def train_score_network(score_net, x_init, t_init, t_final, n_steps, epochs=50, lr=1e-3, noise_std=0.1, batch_size=4096):
    optimizer = optim.Adam(score_net.parameters(), lr=lr)
    
    # 1. Generazione Dati SDE
    x_samples, t_samples = euler_maruyama_sampler(x_init, t_init, t_final, n_steps)
    n_steps_total, N, d = x_samples.shape
    
    # Flattening
    x_train = x_samples.reshape(-1, d)
    t_train = t_samples.repeat(N, 1).T.reshape(-1, 1)
    
    # Inseriamo i dati in un Dataset e usiamo il DataLoader per dividerli in batch 
    dataset = TensorDataset(x_train, t_train)
    # shuffle=True è cruciale per mescolare i frame temporali ed evitare che la rete veda solo t=0 o t=5 in sequenza!
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    score_net.train()
    epoch_bar = tqdm(range(epochs))
    
    for epoch in epoch_bar:
        epoch_loss = 0.0
        
        # Iteriamo su tutti i mini-batch
        for batch_x, batch_t in dataloader:
            optimizer.zero_grad()
            
            # Calcoliamo la loss SOLO sul mini-batch (es. 4096 campioni)
            loss = score_matching_loss(score_net, batch_x, batch_t, noise_std)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
        # Calcoliamo la loss media dell'epoca
        avg_loss = epoch_loss / len(dataloader)
        epoch_bar.set_description(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")
        
    return score_net

dim = 2          # Dimensione spaziale (d)
N = 4096         # Numero di particelle simulate
n_steps = 200    # Passi temporali
t_init = 0.0
t_final = 3.0

# Inizializziamo le particelle (N, d)
x_init = torch.randn(N, dim)*0.5

# Inizializziamo la rete
score_net = ScoreNetwork(spatial_dim=dim, hidden_dim=256)

# Avviamo l'addestramento!
trained_net = train_score_network(
    score_net, 
    x_init, t_init, t_final, n_steps, 
    epochs=50,
    noise_std=0.1,
    lr=1e-3
)

# save the model
torch.save(trained_net.state_dict(), "score_network.pth")

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
#| code-summary: "Score field visualization"
#| echo: false
"""
# plot score-matching like a vectorial space
trained_net = ScoreNetwork(spatial_dim=dim, hidden_dim=256)
trained_net.load_state_dict(torch.load("score_network.pth"))
trained_net.eval()

# generate a grid of points in the 2D space in the time range [t_init, t_final]
x = torch.linspace(-2, 2, 40)  # has shape Nx
y = torch.linspace(-2, 2, 40)  # has shape Ny
t = torch.linspace(t_init, t_final, int((t_final-t_init)*30))  # has shape Nt
X, Y = torch.meshgrid(x, y, indexing="ij")  # have shape (Nx, Ny)
X = X.repeat(len(t), 1, 1)  # has shape (Nt, Nx, Ny)
Y = Y.repeat(len(t), 1, 1)  # has shape (Nt, Nx, Ny)
T = t[:,None].repeat(1, len(x)*len(y))  # has shape (Nt, Nx*Ny)
X = X.reshape(-1, 1)
Y = Y.reshape(-1, 1)
T = T.reshape(-1, 1)

# compute the score
net_in = torch.cat([X, Y, T], dim=-1)  # has shape (Nt*Nx*Ny, dim)
net_out = trained_net(net_in)  # has shape (Nt*Nx*Ny, dim)
net_out = net_out.reshape(t.shape[0], x.shape[0], y.shape[0], -1)  # has shape (Nt, Nx, Ny, dim)

del X, Y, T

# show the score field as a quiver plot in an animation
fig, ax = plt.subplots()
Xg, Yg = torch.meshgrid(x, y, indexing="ij")
Xg = Xg.cpu().numpy()
Yg = Yg.cpu().numpy()

U = net_out[..., 0].detach().cpu().numpy()
V = net_out[..., 1].detach().cpu().numpy()

r = np.sqrt(U**2 + V**2)
r = 1.0/(r + 1)
U = U * r
V = V * r

ax.set_xlim(x.min().item(), x.max().item())
ax.set_ylim(y.min().item(), y.max().item())
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Score field")
ax.set_aspect("equal", adjustable="box")

quiver = ax.quiver(Xg, Yg, U[0], V[0], pivot="mid", angles="xy", scale_units="xy", scale=5)
time_text = ax.text(
    0.02, 0.98, "",
    transform=ax.transAxes,
    va="top",
    color="black",
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=2)
)

def update(frame):
    quiver.set_UVC(U[frame], V[frame])
    time_text.set_text(f"t = {t[frame].item():.2f}")
    return quiver, time_text

ani = FuncAnimation(fig, update, frames=len(t), interval=1000/30, blit=False)

ani.save("score_field.gif", writer=PillowWriter(fps=30), dpi=150)
plt.close(fig)
"""
pass

<img src="score_field.gif" alt="ScorePINN simulation animation" width="600">

## Score-PINN idea

The authors insert the evolution of the score by using the FPE as follows:
$$
\begin{align*}
    \partial_t \textbf{s} &= \partial_t \nabla_\textbf{x} \log u(\textbf{x},t) \\
    &= \nabla_\textbf{x} \partial_t \log u(\textbf{x},t) \\
    &= \nabla_\textbf{x} \left[\frac{\partial_t u(\textbf{x},t)}{u(\textbf{x},t)}\right] \\
    &= \nabla_\textbf{x} \left[\frac{\nabla_\textbf{x}\cdot\left(u\left(\textbf{w}+\frac12\Sigma \textbf{s}\right)\right)}{u}\right] \\
    &\text{where $\textbf{w}=-\textbf{f}+\frac12\nabla_\textbf{x}\cdot\Sigma$}\\
    &= \nabla_\textbf{x} \left[\frac{\nabla_\textbf{x}u\cdot\left(\textbf{w}+\frac12\Sigma \textbf{s}\right)+u\nabla_\textbf{x}\cdot\left(\textbf{w}+\frac12\Sigma \textbf{s}\right)}{u}\right] \\
    &= \nabla_\textbf{x} \left[\textbf{s}\cdot\left(\textbf{w}+\frac12\Sigma \textbf{s}\right)+\nabla_\textbf{x}\cdot\left(\textbf{w}+\frac12\Sigma \textbf{s}\right)\right]
\end{align*}
$$